# IHSG Tail-Risk Monitor (EVT): feasibility probe

**Status: exploratory feasibility check only.** This notebook tests whether the local daily IHSG history can support a transparent one-dimensional extreme-loss monitor. It does not build a production feature, a maturity–moneyness surface, an ML emulator, or a trading signal.

## Research question
Can the available daily IHSG log-return sample support stable estimates of left-tail severity using simple empirical, Hill, and Generalized Pareto Distribution (GPD) summaries? The proposed screen is falsified for product use if estimates change materially across reasonable thresholds, have very wide uncertainty, or depend on a small number of observations.

## Scope and constraints
- Source: local Sectors.app snapshot already assembled in `docs/prototypes/market-overview-data.json`; no network or API calls.
- Frequency: daily closes only. No option maturity, strike/moneyness, implied volatility, or intraday series is assumed.
- Estimation: Hill tail-index sensitivity and GPD exceedance fits at several quantiles; a moving-block bootstrap is a rough dependence-aware uncertainty diagnostic.
- No prediction, backtest, strategy, or investment recommendation is produced.

## Roadmap
1. Load and audit provenance, date coverage, duplicate dates, and missing/invalid closes.
2. Derive daily log returns and loss magnitudes from consecutive completed daily closes.
3. Compare empirical and Gaussian baseline tail thresholds with GPD fits across thresholds.
4. Check Hill/GPD sensitivity and rough 5-session moving-block bootstrap uncertainty.
5. Decide whether a Market Overview prototype is defensible, and list what it cannot claim.

**Data guard:** all transformations are descriptive and use only the loaded historical snapshot. No external data, model training, or future return labels are used.

In [1]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.stats import genpareto, norm
from IPython.display import display

SEED = 20260929
rng = np.random.default_rng(SEED)
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__})

{'python': '3.14.2', 'numpy': '2.4.1', 'pandas': '2.2.2', 'scipy': '1.17.1'}


## 1. Load the local snapshot and establish provenance
The notebook searches a few repository-relative locations and stops with an actionable message if the static snapshot is not present. The manifest fields below are carried from the Sectors-derived local artifact; they are not independently re-fetched here.

In [2]:
candidate_paths = [
    Path("docs/prototypes/market-overview-data.json"),
    Path("../docs/prototypes/market-overview-data.json"),
    Path("market-overview-data.json"),
]
snapshot_path = next((p.resolve() for p in candidate_paths if p.is_file()), None)
if snapshot_path is None:
    raise FileNotFoundError("Attach or place the existing local Sectors snapshot at docs/prototypes/market-overview-data.json (or next to this notebook as market-overview-data.json). No network fallback is used.")

snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))
ihsg = snapshot.get("ihsg", {})
if not ihsg.get("ok") or not ihsg.get("series"):
    raise ValueError("The local snapshot does not contain a usable IHSG daily series.")

manifest = {key: ihsg.get(key) for key in ["provider", "source_class", "endpoint", "data_quality", "observation_count", "coverage_start", "coverage_end", "retrieved_at"]}
manifest["local_snapshot"] = str(snapshot_path)
manifest["source_files"] = ihsg.get("source_files", [])
display(pd.Series(manifest, name="IHSG snapshot provenance").to_frame())

,IHSG snapshot provenance
provider,Sectors.app
source_class,sectors_source_data
endpoint,/v2/index-daily/ihsg/
data_quality,VERIFIED
observation_count,708
coverage_start,2023-09-25
coverage_end,2026-09-24
retrieved_at,2026-09-27T16:43:12.252647+00:00
local_snapshot,/Users/rasyaaudreabramantyawijaya/Downloads/Ha...
source_files,[data/raw/sectors/index_daily/ihsg/ihsg_2023-0...


## 2. Schema and quality audit
Each observation is one dated index close. Daily log return is `log(close_t / close_(t-1))`; the left tail is expressed as positive loss `-return` for readable exceedance thresholds. This probe requires unique, ordered dates and positive closes. It reports rather than silently repairs duplicates or invalid records.

In [3]:
prices = pd.DataFrame(ihsg["series"])
required = {"date", "price"}
missing_columns = required - set(prices.columns)
if missing_columns:
    raise ValueError(f"Missing required IHSG fields: {sorted(missing_columns)}")
prices = prices[["date", "price"]].copy()
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
prices["price"] = pd.to_numeric(prices["price"], errors="coerce")
audit = {
    "rows": len(prices),
    "duplicate_dates": int(prices["date"].duplicated().sum()),
    "invalid_dates": int(prices["date"].isna().sum()),
    "missing_or_non_numeric_prices": int(prices["price"].isna().sum()),
    "non_positive_prices": int((prices["price"] <= 0).sum()),
    "reported_observations": ihsg.get("observation_count"),
}
display(pd.Series(audit, name="audit").to_frame())
if any(audit[key] for key in ["duplicate_dates", "invalid_dates", "missing_or_non_numeric_prices", "non_positive_prices"]):
    raise ValueError("Data-quality guard failed; inspect the audit before estimating tails.")
prices = prices.sort_values("date").reset_index(drop=True)
if not prices["date"].is_monotonic_increasing:
    raise ValueError("Dates are not strictly ordered after sorting.")
prices["log_return"] = np.log(prices["price"] / prices["price"].shift(1))
prices["loss"] = -prices["log_return"]
returns = prices.dropna(subset=["log_return"]).copy()
losses = returns["loss"].to_numpy(dtype=float)
print(f"Return observations: {len(returns)} | {returns.date.min().date()} to {returns.date.max().date()} | daily close-to-close")
display(returns[["date", "price", "log_return", "loss"]].head())

/var/folders/92/6nhl2gvs7t39bfcqvytg33280000gn/T/ipykernel_4954/860723071.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
/var/folders/92/6nhl2gvs7t39bfcqvytg33280000gn/T/i

,audit
rows,708
duplicate_dates,0
invalid_dates,0
missing_or_non_numeric_prices,0
non_positive_prices,0
reported_observations,708


Return observations: 707 | 2023-09-26 to 2026-09-24 | daily close-to-close


/var/folders/92/6nhl2gvs7t39bfcqvytg33280000gn/T/ipykernel_4954/860723071.py:23: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  prices["log_return"] = np.log(prices["price"] / prices["price"].shift(1))
/var/folders/92/6nhl2gvs7t39bfcqvytg3328

,date,price,log_return,loss
1,2023-09-26,6923.800,-0.010714,0.010714
2,2023-09-27,6937.828,0.002024,-0.002024
3,2023-09-29,6939.892,0.000297,-0.000297
4,2023-10-02,6961.459,0.003103,-0.003103
5,2023-10-03,6940.887,-0.002960,0.002960


## 3. Descriptive baseline before EVT
The empirical quantiles are a simple, transparent baseline. A Gaussian quantile is shown only as a distributional comparator; it is not treated as the truth. The sample is about three years long, so a small tail sample is expected even when the daily history looks visually long.

In [4]:
loss_mean, loss_sd = float(np.mean(losses)), float(np.std(losses, ddof=1))
baseline_rows = []
for probability in [0.90, 0.95, 0.975, 0.99]:
    empirical = float(np.quantile(losses, probability))
    gaussian = float(norm.ppf(probability, loc=loss_mean, scale=loss_sd))
    baseline_rows.append({"tail_probability": probability, "empirical_loss_threshold": empirical, "gaussian_loss_threshold": gaussian, "empirical_exceedances": int((losses > empirical).sum())})
baseline = pd.DataFrame(baseline_rows)
display(baseline.style.format({"tail_probability": "{:.1%}", "empirical_loss_threshold": "{:.3%}", "gaussian_loss_threshold": "{:.3%}"}))
print(f"Daily loss mean={loss_mean:.4%}; sample standard deviation={loss_sd:.4%}; worst observed daily log return={returns.log_return.min():.4%}")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(prices.date, prices.price, color="#28b6c8", lw=1.4)
axes[0].set(title="IHSG daily close (local snapshot)", ylabel="Index points")
axes[0].grid(alpha=.2)
axes[1].hist(losses, bins=45, color="#527895", alpha=.85)
for probability, color in [(0.90, "#d0a33c"), (0.95, "#df6d72"), (0.975, "#ae79cf")]:
    axes[1].axvline(np.quantile(losses, probability), color=color, ls="--", label=f"Empirical {probability:.1%}")
axes[1].set(title="Daily loss distribution and empirical thresholds", xlabel="Loss = negative daily log return", ylabel="Sessions")
axes[1].legend(frameon=False)
axes[1].grid(alpha=.2)
plt.tight_layout()
plt.show()

,tail_probability,empirical_loss_threshold,gaussian_loss_threshold,empirical_exceedances
0,90.0%,1.375%,1.621%,71
1,95.0%,1.941%,2.076%,36
2,97.5%,2.763%,2.471%,18
3,99.0%,3.899%,2.930%,8


Daily loss mean=0.0149%; sample standard deviation=1.2530%; worst observed daily log return=-8.2319%


<Figure size 1200x400 with 2 Axes>

## 4. GPD threshold sensitivity
For each empirical loss threshold `u`, fit a Generalized Pareto Distribution to positive excesses `loss - u` where `loss > u`, fixing location at zero. The shape `xi` describes tail curvature; `1/xi` is displayed only when `xi > 0`, where the fitted GPD has a Pareto-type tail index. Threshold choice is a major modeling decision, so the estimates and exceedance counts are shown together.

In [5]:
threshold_rows = []
for probability in [0.90, 0.925, 0.95, 0.975]:
    threshold = float(np.quantile(losses, probability))
    excess = losses[losses > threshold] - threshold
    if len(excess) >= 8:
        shape, loc, scale = genpareto.fit(excess, floc=0)
        alpha = 1.0 / shape if shape > 0 else np.nan
    else:
        shape, scale, alpha = np.nan, np.nan, np.nan
    threshold_rows.append({"quantile": probability, "loss_threshold": threshold, "exceedance_count": len(excess), "gpd_shape_xi": shape, "gpd_scale": scale, "implied_tail_alpha_if_xi_positive": alpha})
gpd_sensitivity = pd.DataFrame(threshold_rows)
display(gpd_sensitivity.style.format({"quantile": "{:.1%}", "loss_threshold": "{:.3%}", "gpd_shape_xi": "{:.3f}", "gpd_scale": "{:.4%}", "implied_tail_alpha_if_xi_positive": "{:.2f}"}))


,quantile,loss_threshold,exceedance_count,gpd_shape_xi,gpd_scale,implied_tail_alpha_if_xi_positive
0,90.0%,1.375%,71,0.220,0.8332%,4.55
1,92.5%,1.641%,53,0.293,0.8009%,3.42
2,95.0%,1.941%,36,0.267,0.9399%,3.74
3,97.5%,2.763%,18,-0.020,1.4883%,nan


## 5. Hill estimator and threshold stability
The Hill estimator assumes a regularly varying, effectively unbounded positive tail. Daily index losses are bounded below by a total loss of 100%, so Hill is a diagnostic rather than an automatic model choice. Its sensitivity to the number of tail observations `k` is itself a feasibility result. Calendar-year tail counts are included to expose how little data is available for a regime-by-regime surface.

In [6]:
positive_losses = np.sort(losses[losses > 0])[::-1]
hill_rows = []
for probability in [0.90, 0.95, 0.975]:
    k = int((losses > np.quantile(losses, probability)).sum())
    if 0 < k < len(positive_losses):
        boundary = positive_losses[k]
        hill_mean = float(np.mean(np.log(positive_losses[:k] / boundary)))
        alpha = 1.0 / hill_mean if hill_mean > 0 else np.nan
    else:
        alpha = np.nan
    hill_rows.append({"tail_quantile": probability, "k_positive_losses": k, "hill_alpha": alpha})
hill_sensitivity = pd.DataFrame(hill_rows)
display(hill_sensitivity.style.format({"tail_quantile": "{:.1%}", "hill_alpha": "{:.2f}"}))

annual_rows = []
for year, group in returns.groupby(returns.date.dt.year):
    annual_loss = group.loss.to_numpy()
    u = float(np.quantile(annual_loss, 0.95))
    n_exc = int((annual_loss > u).sum())
    xi = np.nan
    if n_exc >= 15:
        xi, _, _ = genpareto.fit(annual_loss[annual_loss > u] - u, floc=0)
    annual_rows.append({"year": int(year), "sessions": len(group), "empirical_95pct_loss": u, "95pct_exceedances": n_exc, "gpd_xi_only_if_n_ge_15": xi})
annual_stability = pd.DataFrame(annual_rows)
display(annual_stability.style.format({"empirical_95pct_loss": "{:.3%}", "gpd_xi_only_if_n_ge_15": "{:.3f}"}))

,tail_quantile,k_positive_losses,hill_alpha
0,90.0%,71,2.08
1,95.0%,36,2.32
2,97.5%,18,2.46


,year,sessions,empirical_95pct_loss,95pct_exceedances,gpd_xi_only_if_n_ge_15
0,2023,66,1.155%,4,nan
1,2024,237,1.432%,12,nan
2,2025,232,1.916%,12,nan
3,2026,172,3.475%,9,nan


## 6. Dependence-aware uncertainty diagnostic
A 5-session moving-block bootstrap resamples contiguous blocks of daily losses, then re-estimates the 95% threshold and GPD shape. This is only a rough uncertainty probe: with a short sample and few exceedances, bootstrap intervals may still be unreliable and do not establish stationarity or predictive accuracy.

In [7]:
def moving_block_sample(values, block_length, generator):
    n = len(values)
    starts = generator.integers(0, n - block_length + 1, size=int(np.ceil(n / block_length)))
    sampled = np.concatenate([values[start:start + block_length] for start in starts])
    return sampled[:n]

BOOTSTRAP_REPLICATES = 300
BLOCK_LENGTH = 5
bootstrap_xi = []
bootstrap_rng = np.random.default_rng(SEED)
for _ in range(BOOTSTRAP_REPLICATES):
    sampled = moving_block_sample(losses, BLOCK_LENGTH, bootstrap_rng)
    u = float(np.quantile(sampled, 0.95))
    excess = sampled[sampled > u] - u
    if len(excess) >= 8:
        try:
            xi, _, _ = genpareto.fit(excess, floc=0)
            if np.isfinite(xi):
                bootstrap_xi.append(float(xi))
        except Exception:
            pass
bootstrap_xi = np.asarray(bootstrap_xi)
base_95 = float(np.quantile(losses, 0.95))
base_excess = losses[losses > base_95] - base_95
base_xi, _, base_scale = genpareto.fit(base_excess, floc=0)
bootstrap_summary = pd.DataFrame([{"base_95pct_threshold": base_95, "base_exceedances": len(base_excess), "base_gpd_xi": base_xi, "valid_bootstrap_fits": len(bootstrap_xi), "bootstrap_xi_2.5pct": np.quantile(bootstrap_xi, .025) if len(bootstrap_xi) else np.nan, "bootstrap_xi_97.5pct": np.quantile(bootstrap_xi, .975) if len(bootstrap_xi) else np.nan, "bootstrap_block_length_sessions": BLOCK_LENGTH}])
display(bootstrap_summary.style.format({"base_95pct_threshold": "{:.3%}", "base_gpd_xi": "{:.3f}", "bootstrap_xi_2.5pct": "{:.3f}", "bootstrap_xi_97.5pct": "{:.3f}"}))
if len(bootstrap_xi):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(bootstrap_xi, bins=24, color="#527895", alpha=.85)
    ax.axvline(base_xi, color="#e27679", ls="--", label=f"Full-sample xi={base_xi:.2f}")
    ax.set(title="5-session block-bootstrap distribution of GPD shape", xlabel="GPD shape xi at re-estimated 95% threshold", ylabel="Bootstrap fits")
    ax.legend(frameon=False)
    ax.grid(alpha=.2)
    plt.tight_layout()
    plt.show()

,base_95pct_threshold,base_exceedances,base_gpd_xi,valid_bootstrap_fits,bootstrap_xi_2.5pct,bootstrap_xi_97.5pct,bootstrap_block_length_sessions
0,1.941%,36,0.267,300,-0.494,0.870,5


<Figure size 800x350 with 1 Axes>

## 7. Readout and boundary of a Market Overview feature
A candidate prototype should show daily IHSG tail-risk summaries over time (for example, rolling empirical loss quantiles and an EVT exceedance-fit diagnostic), not a maturity–moneyness surface. The latter requires a compatible options dataset, which this probe does not assume. The summary below is diagnostic only and should not be promoted as a calibrated crash probability or signal.

In [8]:
q95_exceedances = int((losses > np.quantile(losses, .95)).sum())
xi_ci_width = float(np.quantile(bootstrap_xi, .975) - np.quantile(bootstrap_xi, .025)) if len(bootstrap_xi) else np.nan
threshold_xi = gpd_sensitivity.set_index("quantile")["gpd_shape_xi"]
threshold_spread = float(threshold_xi.max() - threshold_xi.min()) if threshold_xi.notna().sum() > 1 else np.nan
print(f"Source observations: {ihsg.get('observation_count')} daily closes | returns: {len(returns)} | span: {returns.date.min().date()}–{returns.date.max().date()}")
print(f"At the full-sample 95th percentile: {q95_exceedances} daily loss exceedances (roughly 5% of return observations).")
print(f"GPD shape threshold spread (90%–97.5%): {threshold_spread:.3f}; 95% 5-session block-bootstrap 95% interval width: {xi_ci_width:.3f}.")
print("Data available for requested 3D option-style surface: maturity=NO, moneyness=NO, implied volatility=NO.")
print("Feasibility verdict: daily index data can support an exploratory one-dimensional historical/EVT tail-risk panel, subject to threshold and uncertainty warnings; it cannot support the pictured maturity–moneyness surface or a PINN/MLP emulator.")

Source observations: 708 daily closes | returns: 707 | span: 2023-09-26–2026-09-24
At the full-sample 95th percentile: 36 daily loss exceedances (roughly 5% of return observations).
GPD shape threshold spread (90%–97.5%): 0.313; 95% 5-session block-bootstrap 95% interval width: 1.365.
Data available for requested 3D option-style surface: maturity=NO, moneyness=NO, implied volatility=NO.
Feasibility verdict: daily index data can support an exploratory one-dimensional historical/EVT tail-risk panel, subject to threshold and uncertainty warnings; it cannot support the pictured maturity–moneyness surface or a PINN/MLP emulator.


## Conclusion and what this probe cannot establish
- It tests data sufficiency and estimator sensitivity only; it does not show that EVT improves a decision, forecasts crashes, or predicts the next loss.
- It does not validate the `−2σ` foreign-flow hypothesis or any trading use.
- The Market Overview prototype now consumes a companion MATLAB-generated historical loss surface from `scripts/build_ihsg_evt_tail_surface.m`; that visual is empirical and is not a maturity–moneyness model. This notebook remains the feasibility analysis and does not itself modify the product page.
- No GPU work, trained estimator, API call, or backtest was run.

## Reproducibility manifest
- Source artifact: local `market-overview-data.json` containing the existing Sectors.app IHSG daily snapshot.
- Snapshot endpoint recorded in data: `/v2/index-daily/ihsg/`; local only, no request made by this notebook.
- Seed: `20260929`; bootstrap: 300 moving-block replicates, block length 5 sessions.
- Thresholds: 90%, 92.5%, 95%, 97.5% empirical daily-loss quantiles.
- No package installation, model training, backtest, or API access.